In [45]:
import math

def extend_shortest_paths(L_prev, W, n):
    L_new = [[math.inf] * n for _ in range(n)]
    
    for i in range(n):
        for j in range(n):
            for k in range(n):
                L_new[i][j] = min(L_new[i][j],
                                  L_prev[i][k] + W[k][j])
    return L_new

In [46]:
def slow_apsp(W):
    n = len(W)
    
    # L^(1) = W
    L = W
    
    for _ in range(n - 2):
        L = extend_shortest_paths(L, W, n)
    
    return L

In [47]:
def slow_apsp_with_layers(W):
    n = len(W)
    L = [None] * n
    L[0] = W

    for r in range(1, n):
        L[r] = extend_shortest_paths(L[r-1], W, n)

    return L

In [48]:
def faster_apsp(W):
    n = len(W)
    
    L = W
    m = 1
    
    while m < n - 1:
        L = extend_shortest_paths(L, L, n)  # square it
        m *= 2
    
    return L

In [49]:
def faster_apsp_with_layers(W):
    n = len(W)
    layers = {1: W}
    
    r = 1
    while r < n - 1:
        layers[2*r] = extend_shortest_paths(layers[r], layers[r], n)
        r *= 2
    
    return layers

In [50]:
"""
Slow APSP:
Let's try paths with 1 more edge each time.

Faster APSP:
Let's double how far we can go each time.

Any shortest path has at most n−1 edges, so:

Slow: takes n−1 iterations
Faster: reaches ≥ n−1 using doubling

| Method | Progression  | # Steps  |
| -------| ------------ | -------- |
| Slow   | L¹ → L² → L³ | O(n)     |
| Faster | L¹ → L² → L⁴ | O(log n) |

GRAPH:

0 --1--> 1 --2--> 2
 \--------------5--->

This means:
0→1 costs 1
1→2 costs 2
0→2 costs 5 (direct edge, but not optimal)

INITIAL MATRIX W (L^(1)):
[0   1   5]
[∞   0   2]
[∞   ∞   0]


STEP: W ⊗ W (this is where improvement happens)

We try all "middle nodes k" for each pair (i, j).

Focus on entry (0, 2):

Try k = 0:
0→0 + 0→2 = 0 + 5 = 5

Try k = 1:
0→1 + 1→2 = 1 + 2 = 3 BEST PATH FOUND

Try k = 2:
0→2 + 2→2 = 5 + 0 = 5


RESULTING MATRIX (L^(2)):
[0   1   3]
[∞   0   2]
[∞   ∞   0]


WHAT JUST HAPPENED (INTUITION):
The matrix multiplication is not “multiplying numbers”.
It is testing ALL possible middle stops k.

So instead of only using the direct edge 0→2 = 5,
it discovers a better two-step route:
0 → 1 → 2 = 1 + 2 = 3

This is why the value changes from 5 → 3.

---

WHY THIS ENABLES FAST APSP:

Slow version:
- builds paths like a staircase
- 1 edge → 2 edges → 3 edges → 4 edges

Fast version:
- combines whole blocks of paths
- 1 edge → 2 edges → 4 edges → 8 edges

Because each multiplication already considers ALL middle nodes,
it “jumps” path length instead of stepping one edge at a time.
"""

'\nSlow APSP:\nLet\'s try paths with 1 more edge each time.\n\nFaster APSP:\nLet\'s double how far we can go each time.\n\nAny shortest path has at most n−1 edges, so:\n\nSlow: takes n−1 iterations\nFaster: reaches ≥ n−1 using doubling\n\n| Method | Progression  | # Steps  |\n| -------| ------------ | -------- |\n| Slow   | L¹ → L² → L³ | O(n)     |\n| Faster | L¹ → L² → L⁴ | O(log n) |\n\nGRAPH:\n\n0 --1--> 1 --2--> 2\n \\--------------5--->\n\nThis means:\n0→1 costs 1\n1→2 costs 2\n0→2 costs 5 (direct edge, but not optimal)\n\nINITIAL MATRIX W (L^(1)):\n[0   1   5]\n[∞   0   2]\n[∞   ∞   0]\n\n\nSTEP: W ⊗ W (this is where improvement happens)\n\nWe try all "middle nodes k" for each pair (i, j).\n\nFocus on entry (0, 2):\n\nTry k = 0:\n0→0 + 0→2 = 0 + 5 = 5\n\nTry k = 1:\n0→1 + 1→2 = 1 + 2 = 3   ← BEST PATH FOUND\n\nTry k = 2:\n0→2 + 2→2 = 5 + 0 = 5\n\n\nRESULTING MATRIX (L^(2)):\n[0   1   3]\n[∞   0   2]\n[∞   ∞   0]\n\n\nWHAT JUST HAPPENED (INTUITION):\n\nThe matrix multiplication i